In [1]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"

### Interacting with the MLflow tracking server

The MlflowClient object allows us to interact with...

+ an MLflow Tracking Server that creates and manages experiments and runs.
+ an MLflow Registry Server that creates and manages registered models and model versions.
To instantiate it we need to pass a tracking URI and/or a registry URI

In [28]:
client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

client.search_experiments()

[<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1786723449402, experiment_id='2', last_update_time=1786723449402, lifecycle_stage='active', name='new-experiment', tags={}>,
 <Experiment: artifact_location='gs://mlops-bucket-hashan-224322/1', creation_time=1786664617927, experiment_id='1', last_update_time=1786664617927, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}>,
 <Experiment: artifact_location='gs://$GCP_BUCKET/0', creation_time=1786645869124, experiment_id='0', last_update_time=1786645869124, lifecycle_stage='active', name='Default', tags={}>]

In [ ]:
client.create_experiment(name="new-experiment")

In [30]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='1',
    filter_string="metrics.rmse < 7 AND tags.estimator_name != ''",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=10,
    order_by=["metrics.rmse ASC"]
)

In [31]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: 8b55439fa40348dc81064a0ec5e5c6d5, rmse: 5.4421
run id: 2504e68c30b244fcb58c140e29efdc9c, rmse: 5.4757
run id: 14d51990034245579bc6e3bbf7040cad, rmse: 5.7611


### Interacting with the Model Registry

In this section We will use the MlflowClient instance to:
+ Register a new model for the experiment nyc-taxi-regressor
+ Retrieve the latests versions of the model nyc-taxi-regressor and check that new versions were created.
+ Set `champion` and `challenger` versions of the model

In [32]:
run_id = "8b55439fa40348dc81064a0ec5e5c6d5"
experiment_id = "1"

model_list = client.search_logged_models(
    experiment_ids=[experiment_id],
    filter_string=f"source_run_id = '{run_id}'",
    max_results=1000,
)

for model in model_list:
    print(model)

LoggedModel(artifact_location='gs://mlops-bucket-hashan-224322/1/models/m-ad0e6aa494a94d78bada9725d6ae9479/artifacts', creation_timestamp=1786725603101, experiment_id='1', last_updated_timestamp=1786725612140, model_id='m-ad0e6aa494a94d78bada9725d6ae9479', model_type='', model_uri='models:/m-ad0e6aa494a94d78bada9725d6ae9479', name='model', source_run_id='8b55439fa40348dc81064a0ec5e5c6d5', status=<LoggedModelStatus.READY: 'READY'>, status_message='')


In [39]:
registered_model_name="nyc-taxi-regressor"
registered_model = client.create_registered_model(name=registered_model_name)

In [40]:
champion_model = client.create_model_version(
    name=registered_model.name,
    source=selected_model.model_uri,
    run_id=selected_model.source_run_id
)
client.set_registered_model_alias(
    name=registered_model.name,
    alias="champion",
    version=champion_model.version,
)

challenger_model = client.create_model_version(
    name=registered_model.name,
    source=selected_model.model_uri,
    run_id=selected_model.source_run_id
)
client.set_registered_model_alias(
    name=registered_model.name,
    alias="challenger",
    version=challenger_model.version,
)

2026/08/14 17:57:27 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: nyc-taxi-regressor, version 1
2026/08/14 17:57:27 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: nyc-taxi-regressor, version 2


In [41]:
def print_model_info(name):

    models = client.search_model_versions(
        filter_string=f"name = '{name}'",
        order_by=["version_number ASC"],
    )
    
    print(f"Model name: '{name}'")
    
    for model in models:
        version_info = client.get_model_version(
            name=model.name,
            version=model.version,
        )
    
        print(
            f"version={version_info.version}, "
            f"aliases={version_info.aliases}, "
            f"source={version_info.source}"
        )

In [42]:
print_model_info(registered_model.name)

Model name: 'nyc-taxi-regressor'
version=1, aliases=['champion'], source=models:/m-ad0e6aa494a94d78bada9725d6ae9479
version=2, aliases=['challenger'], source=models:/m-ad0e6aa494a94d78bada9725d6ae9479


### Comparing versions and selecting the new champion model

The idea is to simulate the scenario in which a deployment engineer has to interact with the model registry to decide whether to update the model version that is in production or not.

These are the steps:
+ Load the test dataset, which corresponds to the NYC Green Taxi data from the month of March 2023.
+ Download the DictVectorizer that was fitted using the training data and saved to MLflow as an artifact, and load it with pickle.
+ Preprocess the test set using the DictVectorizer so we can properly feed the regressors.
+ Make predictions on the test set using the model versions that are currently labeled as `challnger` and `champion`, and compare their performance.
+ Based on the results, set the new `champion` model version accordingly.

In [43]:
from sklearn.metrics import root_mean_squared_error
import pandas as pd
import pickle


def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df


def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')
    return dv.transform(train_dicts)


def test_model(version_info, df, dv):
    client.download_artifacts(run_id=version_info.run_id, path='preprocessor', dst_path='.')

    with open("preprocessor/preprocessor.b", "rb") as f_in:
        dv = pickle.load(f_in)

    X_test = preprocess(df, dv)

    target = "duration"
    y_test = df[target].values

    model = mlflow.pyfunc.load_model(version_info.source)
    y_pred = model.predict(X_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}

In [44]:
df = read_dataframe("data/green_tripdata_2023-03.parquet")

In [45]:
challenger_version_info = client.get_model_version_by_alias(
    name=registered_model_name,
    alias="challenger",
)

champion_version_info = client.get_model_version_by_alias(
    name=registered_model_name,
    alias="champion",
)

In [47]:
challenger_version_info

<ModelVersion: aliases=['challenger'], creation_timestamp=1786730247275, current_stage='None', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1786730247275, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id='8b55439fa40348dc81064a0ec5e5c6d5', run_link='', source='models:/m-ad0e6aa494a94d78bada9725d6ae9479', status='READY', status_message=None, tags={}, user_id='', version='2'>

In [54]:
# artifacts = client.list_artifacts(run_id=challenger_version_info.run_id)

run = client.get_run(challenger_version_info.run_id)

print(run.info.artifact_uri)
print(run.info.experiment_id)
print(run.info.run_id)

gs://mlops-bucket-hashan-224322/1/8b55439fa40348dc81064a0ec5e5c6d5/artifacts
1
8b55439fa40348dc81064a0ec5e5c6d5


In [48]:
import pickle

client.download_artifacts(run_id=challenger_version_info.run_id, path='preprocessor/preprocessor.b')
# dv = pickle.load(f_in)

# %time test_model(version_info=challenger_version_info, df=df, dv=dv)

/opt/anaconda3/envs/experiment-tracking/lib/python3.9/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/opt/anaconda3/envs/experiment-tracking/lib/python3.9/site-packages/google/oauth2/__init__.py:40: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade your Python version, and then update google-auth.
  warnings.warn(eol_message.format("3.9"), FutureWarning)
/opt/anaconda3/envs/experiment-tracking/lib/python3.9/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python versio

MlflowException: Failed to download artifacts from path 'preprocessor.b', please ensure that the path is correct.

In [110]:
%time test_model(version_info=champion_version_info, df=df, dv=dv)

CPU times: user 7.09 s, sys: 703 ms, total: 7.79 s
Wall time: 7.78 s


{'rmse': 5.716225073043043}

In [111]:
client.set_registered_model_alias(
    name=model_name,
    alias="champion",
    version=challenger_version_info.version,
)

client.delete_registered_model_alias(
    name=model_name,
    alias="challenger",
)

print_model_info(registered_model_name)

Model name: 'nyc-taxi-regressor'
version=1, aliases=[], source=models:/m-42c1b8dfe263496daee0b2355a2cea81
version=2, aliases=[], source=models:/nyc-taxi-regressor/1
version=3, aliases=[], source=models:/nyc-taxi-regressor/2
version=4, aliases=[], source=models:/m-42c1b8dfe263496daee0b2355a2cea81
version=5, aliases=[], source=models:/m-42c1b8dfe263496daee0b2355a2cea81
version=6, aliases=['champion'], source=models:/m-42c1b8dfe263496daee0b2355a2cea81
